# Logs

> Generating / formatting the base logging instance

In [ ]:
#| default_exp mcp

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Iterable, Tuple, Set

import os, sys
import argparse

from mcp.server.fastmcp import FastMCP  # official MCP Python SDK (FastMCP 2.0+)

from nbdev_mcp import __version__
from nbdev_mcp.utils.logs import log
from nbdev_mcp.utils.config import CURRENT_PROJECT
from nbdev_mcp.utils.paths import resolve_selector
from nbdev_mcp.utils.subprocess import watch_notebooks
from nbdev_mcp.resources import add_resources
from nbdev_mcp.tools import (
    add_project_tools, add_env_tools, add_nbdev_tools, 
    add_notebook_editing_tools, add_lint_tools, 
    add_analysis_tools, add_tests_tools
)
from nbdev_mcp.prompts import add_prompts
from nbdev_mcp.tasks import add_task_tools, enable_nbdev_tasks

## HTTP Utils

In [ ]:
#| export
def set_http_path_if_supported(target_path: str) -> bool:
    """Try to set the HTTP mount path on the MCP SDK settings.
    
    Parameters
    ----------
    target_path : str
        The path to set (e.g., '/mcp').
    
    Returns
    -------
    bool
        True if successfully set, False if not supported.
    """
    import mcp
    try:
        mcp.settings.streamable_http_path = target_path  # type: ignore[attr-defined]
        return True
    except Exception:
        try:
            mcp.settings.http_path = target_path  # type: ignore[attr-defined]
            return True
        except Exception:
            return False

## Create the MCP

In [ ]:
#| export
def create_nbdev_mcp(name: str = "mcp.nbdev") -> FastMCP:
    """Create and configure the nbdev MCP server with all resources, tools, and prompts."""
    mcp = FastMCP(name)
    
    # Enable experimental tasks API
    enable_nbdev_tasks(mcp)
    
    # Attach all nbdev-related resources, tools, and prompts
    add_resources(mcp)
    add_project_tools(mcp)
    add_env_tools(mcp)
    add_nbdev_tools(mcp)
    add_notebook_editing_tools(mcp)  # CRITICAL: Notebook editing and workflow tools
    add_prompts(mcp)  # Philosophy prompts must come after tools are registered
    
    # Extensions: linting, analysis, and code generation tools
    add_lint_tools(mcp)
    add_analysis_tools(mcp)
    add_tests_tools(mcp)
    
    # Task-based tools for auditing, deduplication, and refactoring
    add_task_tools(mcp)
    
    return mcp

## `main`

In [ ]:
#| export
def main() -> None:
    """Entry point for the nbdev MCP server CLI."""
    parser = argparse.ArgumentParser(
        description="nbdev MCP server (multi-project, rich output)",
        formatter_class=argparse.RawTextHelpFormatter,
        epilog=(
            "Example: python mcp.nbdev.py --transport http --host 0.0.0.0 --port 8765 --path /mcp\n"
            "Example: nbdev-mcp --watch --project /path/to/project  # Watch mode\n"
            "Note: Ensure 'uvicorn' is installed for custom HTTP server."
        )
    )
    parser.add_argument("--project", help="Path or alias for an nbdev project to load initially.")
    parser.add_argument("--transport", choices=("stdio", "http", "streamable-http"), 
                        default=os.environ.get("NBDEV_MCP_TRANSPORT", "stdio"),
                        help="Transport mode: 'stdio' (for CLI/desktop clients), 'streamable-http' (built-in HTTP on default port/path), or 'http' (custom host/port via Uvicorn).")
    parser.add_argument("--host", default=os.environ.get("NBDEV_MCP_HOST", "127.0.0.1"),
                        help="Host interface to serve on (default: NBDEV_MCP_HOST or 127.0.0.1).")
    parser.add_argument("--port", type=int, default=int(os.environ.get("NBDEV_MCP_PORT", "8000")),
                        help="Port to serve on (default: NBDEV_MCP_PORT or 8000).")
    parser.add_argument("--path", default=os.environ.get("NBDEV_MCP_PATH", "/mcp"),
                        help="Base URL path for HTTP transport (default: NBDEV_MCP_PATH or /mcp).")
    parser.add_argument("-v", "--verbose", action="store_true",
                        help="Enable verbose/debug output for troubleshooting.")
    parser.add_argument("-w", "--watch", action="store_true",
                        help="Watch mode: monitor notebooks and auto-export on changes (requires --project).")
    parser.add_argument("--watch-interval", type=float, default=2.0,
                        help="Watch mode polling interval in seconds (default: 2.0).")
    parser.add_argument("--watch-cmd", default="nbdev_export",
                        help="Command to run on change in watch mode (default: nbdev_export).")
    parser.add_argument("-V", "--version", action="version", version=f"%(prog)s {__version__}")
    args = parser.parse_args()

    # Configure logging based on verbose flag
    if args.verbose:
        import logging
        logging.basicConfig(
            level=logging.DEBUG,
            format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S"
        )
        log.setLevel(logging.DEBUG)
        log.debug("Verbose mode enabled")
        log.debug(f"Python: {sys.version}")
        log.debug(f"nbdev-mcp version: {__version__}")
        log.debug(f"Transport: {args.transport}")
        log.debug(f"Project: {args.project or 'None'}")

    # If an initial project path or alias is provided, set it now
    proj_path = None
    if args.project:
        try:
            proj_path = resolve_selector(args.project)
            CURRENT_PROJECT = proj_path  # set the global current project
            if args.verbose:
                log.debug(f"Initial project set to: {proj_path}")
        except Exception as e:
            log.error(str(e))
            if args.watch:
                sys.exit(1)

    # Watch mode: monitor notebooks and auto-export
    if args.watch:
        if not proj_path:
            log.error("Watch mode requires --project to be specified")
            sys.exit(1)
        watch_notebooks(proj_path, interval=args.watch_interval, on_change=args.watch_cmd)
        return  # Exit after watch mode ends

    # Build the MCP server with all nbdev features
    mcp = create_nbdev_mcp()
    if args.verbose:
        log.debug("MCP server created with all tools, resources, and prompts")

    # Decide transport using structural pattern matching (Python 3.10+)
    default_host, default_port, default_path = "127.0.0.1", 8000, "/mcp"
    using_defaults = (args.host == default_host and args.port == default_port and args.path == default_path)

    match args.transport:
        case "stdio":
            if args.verbose:
                log.debug("Starting MCP server with stdio transport")
            mcp.run(transport="stdio")
        case "streamable-http":
            if using_defaults:
                # Default behavior: run built-in server on 127.0.0.1:8000 at /mcp
                if args.verbose:
                    log.debug(f"Starting MCP server with streamable-http transport on {default_host}:{default_port}{default_path}")
                mcp.run(transport="streamable-http")
            else:
                # Custom host/port for streamable HTTP – use Uvicorn as fallback
                try:
                    import uvicorn
                except ImportError:
                    log.error("uvicorn is required for custom host/port HTTP transport. Please install uvicorn and try again.")
                    sys.exit(1)
                if args.path and args.path != default_path:
                    ok = set_http_path_if_supported(args.path)
                    if not ok:
                        log.warning("Could not set custom HTTP path on this SDK; using default '/mcp'.")
                if args.verbose:
                    log.debug(f"Starting MCP server with streamable-http transport via uvicorn on {args.host}:{args.port}")
                app = mcp.streamable_http_app()
                uvicorn.run(app, host=args.host, port=args.port)
        case "http":
            # Use Uvicorn to serve HTTP on the specified host/port
            try:
                import uvicorn
            except ImportError:
                log.error("uvicorn is required for custom host/port HTTP transport. Please install uvicorn and try again.")
                sys.exit(1)
                ok = set_http_path_if_supported(args.path)
            if args.path and args.path != default_path:
                if not ok:
                    log.warning("Could not set custom HTTP path on this SDK; using default '/mcp'.")
            if args.verbose:
                log.debug(f"Starting MCP server with http transport via uvicorn on {args.host}:{args.port}")
            app = mcp.streamable_http_app()
            uvicorn.run(app, host=args.host, port=args.port)
        case _:
            raise SystemExit(f"Unsupported transport option: {args.transport!r}")

## Next

In [ ]:
#| export


## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()